In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

#################################################################################################
# Database Configuration
#################################################################################################

ENGINE = os.getenv("DB_ENGINE")
NAME = os.getenv("DB_NAME")
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")

DATA_DIR = "data/output"

In [2]:
#################################################################################################
# Database Connection
#################################################################################################

# Create a SQLAlchemy engine to connect to the PostgreSQL database.
engine = create_engine(
    f"{ENGINE}://{USER}:{PASSWORD}@{HOST}:{PORT}/{NAME}",
    connect_args={
        "keepalives": 1,
        "keepalives_idle": 30,
        "keepalives_interval": 10,
        "keepalives_count": 5,
    }
)

# Load Parquet Files
users = pd.read_parquet(f"{DATA_DIR}/user.parquet")
items = pd.read_parquet(f"{DATA_DIR}/item.parquet")
reviews = pd.read_parquet(f"{DATA_DIR}/user-item-interaction.parquet")

In [3]:
# Write each DataFrame to its corresponding table in PostgreSQL.
# NOTE: if_exists="replace" drops and recreates the table on every run, ensuring the database
#       always reflects the latest parquet data.
# NOTE: index=False prevents pandas from writing the DataFrame index as a column.
users.to_sql("Users", engine, if_exists="replace", index=False)
items.to_sql("Items", engine, if_exists="replace", index=False)
reviews.to_sql("Reviews", engine, if_exists="replace", index=False, chunksize=1000)

print("Complete!")


Complete!
